<a href="https://colab.research.google.com/github/jano87/diplomado-MLunsupervised/blob/main/AllCategoriesFullDiscoveryMLC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MercadoLibre Chile: which categories have the most buyer demand? (v3: trends + listings)

**Goal:** find which of MercadoLibre Chile's (`MLC`) ~2,900 level-3 categories attract the most buyer demand, without listing them beforehand, then group them into market profiles with PCA + k-means.

## What the API allows (tested 22 Oct 2026 with a read-only app)
| Signal | Endpoint | Status |
|---|---|---|
| Category tree + listing counts | `/categories/{id}` | ✅ public |
| Site-wide top 50 trending searches | `/trends/MLC` | ✅ token |
| Top ≤50 trending searches **per category** | `/trends/MLC/{cat}` | ✅ token (404 = no public trends) |
| Keyword → category | `/sites/MLC/domain_discovery/search` | ✅ public |
| Best-seller rankings `/highlights` | | ❌ 403 PolicyAgent (app and user tokens) |
| Listing search `/sites/MLC/search` | | ❌ 403 since April 2025 |

MercadoLibre doesn't expose sales for other sellers, so this notebook measures **buyer search demand** and **seller supply**, not units sold.

## Features per level-3 category
* **`trend_depth`** (0–50): how many trending searches the category has. Busy categories fill all 50 slots; quiet ones return 404 (= 0).
* **`site_trend_score`**: the site-wide top 50 searches, mapped to categories and weighted by rank (1/√position).
* **`root_trend_score`**: the same for each top-level category's top 50 list. This compares categories *within* a vertical.
* **`log_listings`**, plus **`listing_growth`** from day 2: supply size and sellers entering.
* **`demand_vs_supply`**: trend depth above or below what the listing count predicts. Positive = more searching than the supply would suggest.
* **`keyword_turnover`** (day 2+): share of today's trending searches that are new. High = fast-moving or seasonal demand.

**Run it once a day.** Day 1 gives a snapshot; day 2+ adds growth and turnover.

## 0 · Credentials (one time)
In Colab's 🔑 **Secrets** panel add `ML_CLIENT_ID` (App ID) and `ML_CLIENT_SECRET` (Client Secret), with notebook access on.
Outside Colab, set the same names as environment variables, or set `ML_ACCESS_TOKEN` to an existing token.
**Never write the secret into the notebook.** This repo is public.

In [ ]:
# --- CELL 1: SETUP & DATABASE ---
import os, time, json, math, sqlite3, threading, warnings
from datetime import datetime
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor
import requests
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    BASE_DIR, IN_COLAB = '/content/drive/MyDrive', True
except ImportError:
    BASE_DIR, IN_COLAB = os.environ.get('ML_BASE_DIR', '.'), False

SITE      = 'MLC'
TODAY     = datetime.now(ZoneInfo('America/Santiago')).date().isoformat()   # one snapshot per Chilean day
DB_PATH   = f'{BASE_DIR}/ml_category_demand_v3.db'
ANALYSIS_DEPTH = 3       # level analysed (3 ≈ 2,900 categories; 2 ≈ 440 broader ones)
MAX_WORKERS    = 6       # parallel requests (well under the app's 18,000 requests/hour)
EXCLUDE_ROOTS  = {'MLC1459', 'MLC1743', 'MLC1540'}   # real estate, vehicles, services: not product demand

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.executescript("""
CREATE TABLE IF NOT EXISTS categories (
    snapshot_date TEXT, category_id TEXT, name TEXT, parent_id TEXT, root_id TEXT, l2_id TEXT,
    depth INTEGER, total_items INTEGER, path TEXT, PRIMARY KEY (snapshot_date, category_id));
CREATE TABLE IF NOT EXISTS trends (
    snapshot_date TEXT, scope TEXT, keyword TEXT, position INTEGER,
    PRIMARY KEY (snapshot_date, scope, keyword));
CREATE TABLE IF NOT EXISTS trend_status (          -- one row per scope per day (also stores 404 = 0 trends)
    snapshot_date TEXT, scope TEXT, n_keywords INTEGER, PRIMARY KEY (snapshot_date, scope));
CREATE TABLE IF NOT EXISTS keyword_map (keyword TEXT PRIMARY KEY, category_id TEXT, mapped_on TEXT);
CREATE TABLE IF NOT EXISTS category_paths (category_id TEXT PRIMARY KEY, path_ids TEXT);
""")
conn.commit()
DB_LOCK = threading.Lock()
print(f"DB ready: {DB_PATH} | snapshot date: {TODAY}")

In [ ]:
# --- CELL 2: API CLIENT (token, retries, thread-safe) ---
def _secret(name):
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

class MeliClient:
    API = 'https://api.mercadolibre.com'

    def __init__(self, client_id=None, client_secret=None, access_token=None):
        self.client_id, self.client_secret = client_id, client_secret
        self.token, self.expires_at = access_token, (time.time() + 5 * 3600 if access_token else 0)
        self.calls, self._lock, self._local = 0, threading.Lock(), threading.local()

    def _session(self):
        if not hasattr(self._local, 's'):
            self._local.s = requests.Session(); self._local.s.headers['Accept'] = 'application/json'
        return self._local.s

    def authenticate(self):
        if not (self.client_id and self.client_secret):
            if self.token:
                return True
            raise RuntimeError("Missing ML_CLIENT_ID / ML_CLIENT_SECRET (Colab Secrets) or ML_ACCESS_TOKEN.")
        # HTTP Basic client auth (verified working); body-params as fallback
        attempts = [dict(auth=(self.client_id, self.client_secret), data={'grant_type': 'client_credentials'}),
                    dict(data={'grant_type': 'client_credentials', 'client_id': self.client_id,
                               'client_secret': self.client_secret})]
        for kw in attempts:
            r = requests.post(f'{self.API}/oauth/token', headers={'content-type': 'application/x-www-form-urlencoded',
                              'accept': 'application/json'}, timeout=30, **kw)
            if r.ok:
                d = r.json()
                self.token = d['access_token']
                self.expires_at = time.time() + int(d.get('expires_in', 21600)) - 300
                print(f"✅ Token OK (expires in {int(d.get('expires_in', 0)) // 3600} h)")
                return True
        raise RuntimeError(f"Token request failed: {r.status_code} {r.text[:200]}")

    def get(self, path, params=None, auth=True, retries=4):
        """Returns (status_code, json_or_None)."""
        if auth and time.time() > self.expires_at:
            with self._lock:
                if time.time() > self.expires_at:
                    self.authenticate()
        for attempt in range(retries):
            h = {'Authorization': f'Bearer {self.token}'} if auth else {}
            try:
                r = self._session().get(self.API + path, params=params, headers=h, timeout=30)
            except requests.RequestException:
                time.sleep(2 ** attempt); continue
            self.calls += 1
            if r.status_code == 200:
                return 200, r.json()
            if r.status_code == 401 and auth and attempt == 0:
                with self._lock:
                    self.expires_at = 0
                    self.authenticate()
                continue
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(1.5 * 2 ** attempt); continue
            return r.status_code, None
        return None, None

    def pmap(self, fn, items, workers=MAX_WORKERS, label=''):
        out, n = [], len(items)
        with ThreadPoolExecutor(workers) as ex:
            for i, res in enumerate(ex.map(fn, items), 1):
                out.append(res)
                if label and (i % 500 == 0 or i == n):
                    print(f"   {label}: {i}/{n}")
        return out

ml = MeliClient(_secret('ML_CLIENT_ID'), _secret('ML_CLIENT_SECRET'), _secret('ML_ACCESS_TOKEN'))
ml.authenticate()

# Quick probe: what is reachable today?
for name, path, auth in [('category tree', '/categories/MLC1051', False), ('root list', f'/sites/{SITE}/categories', True),
                         ('site trends', f'/trends/{SITE}', True), ('category trends', f'/trends/{SITE}/MLC1055', True),
                         ('best sellers (expected ❌)', f'/highlights/{SITE}/category/MLC1055', True)]:
    code_, _ = ml.get(path, auth=auth)
    print(f"{'✅' if code_ == 200 else '❌'} {name:<26} HTTP {code_}")

## 1 · Discover the category tree
Read the tree down to `ANALYSIS_DEPTH`. Listing counts for the deepest level come from the parent's response, so this needs only ~470 calls.

In [ ]:
# --- CELL 3: CATEGORY TREE + LISTING COUNTS ---
def crawl_tree(max_depth):
    code_, roots = ml.get(f'/sites/{SITE}/categories')
    roots = [r for r in (roots or []) if r['id'] not in EXCLUDE_ROOTS]
    rows, level = [], [(r['id'], None, r['id'], None, 1) for r in roots]
    while level:
        results = ml.pmap(lambda x: ml.get(f'/categories/{x[0]}', auth=False)[1], level)
        nxt = []
        for (cid, parent, root, l2, depth), d in zip(level, results):
            if not d:
                continue
            path = ' > '.join(p['name'].strip() for p in d.get('path_from_root', []))
            if depth == 1:
                rows.append(dict(category_id=cid, name=d['name'].strip(), parent_id=None, root_id=cid, l2_id=None,
                                 depth=1, total_items=d.get('total_items_in_this_category'), path=path))
            for k in d.get('children_categories', []):
                kid_l2 = k['id'] if depth == 1 else l2
                rows.append(dict(category_id=k['id'], name=k['name'].strip(), parent_id=cid, root_id=root, l2_id=kid_l2,
                                 depth=depth + 1, total_items=k.get('total_items_in_this_category'),
                                 path=f"{path} > {k['name'].strip()}"))
                if depth + 1 < max_depth:
                    nxt.append((k['id'], cid, root, kid_l2, depth + 1))
        level = nxt
    df = pd.DataFrame(rows).drop_duplicates('category_id')
    df.loc[df.depth == 2, 'l2_id'] = df.loc[df.depth == 2, 'category_id']
    return df.assign(snapshot_date=TODAY)

t0 = time.time()
tree = crawl_tree(ANALYSIS_DEPTH)
cols = ['snapshot_date', 'category_id', 'name', 'parent_id', 'root_id', 'l2_id', 'depth', 'total_items', 'path']
with DB_LOCK:
    conn.executemany(f"INSERT OR REPLACE INTO categories ({','.join(cols)}) VALUES ({','.join('?'*len(cols))})",
                     tree[cols].astype(object).where(tree[cols].notna(), None).values.tolist())
    conn.commit()
print(f"🌳 {len(tree):,} categories | by depth: {tree.depth.value_counts().sort_index().to_dict()} | {time.time()-t0:.0f}s")

## 2 · Collect trending searches
* One call per category up to `ANALYSIS_DEPTH`, plus the site-wide list. A 404 means "no public trends" and is stored as 0.
* The keywords on the site-wide and top-level lists are mapped to the category buyers would land in, using `domain_discovery`. Mappings are cached, so later days are cheaper.

In [ ]:
# --- CELL 4: TRENDS FOR EVERY CATEGORY ---
def fetch_trends(scope):
    path = f'/trends/{SITE}' if scope == 'SITE' else f'/trends/{SITE}/{scope}'
    code_, data = ml.get(path)
    if code_ == 200:
        kws = [t['keyword'].strip().lower() for t in data if t.get('keyword')]
        return scope, kws
    return scope, ([] if code_ == 404 else None)        # None = real failure, retried next run

done = {r[0] for r in conn.execute("SELECT scope FROM trend_status WHERE snapshot_date=?", (TODAY,))}
scopes = [s for s in ['SITE'] + tree.category_id.tolist() if s not in done]      # resumable
t0 = time.time()
results = ml.pmap(fetch_trends, scopes, label='trend lists')
failed = 0
with DB_LOCK:
    for scope, kws in results:
        if kws is None:
            failed += 1; continue
        conn.execute("INSERT OR REPLACE INTO trend_status VALUES (?,?,?)", (TODAY, scope, len(kws)))
        conn.executemany("INSERT OR REPLACE INTO trends VALUES (?,?,?,?)",
                         [(TODAY, scope, k, i) for i, k in enumerate(kws, 1)])
    conn.commit()
st_ = pd.read_sql("SELECT n_keywords FROM trend_status WHERE snapshot_date=? AND scope!='SITE'", conn, params=(TODAY,))
print(f"🔎 {len(scopes):,} lists in {time.time()-t0:.0f}s | failures: {failed} | "
      f"categories with 0 trends: {(st_.n_keywords == 0).mean():.0%} | with full 50: {(st_.n_keywords == 50).mean():.0%}")

In [ ]:
# --- CELL 5: MAP SITE + ROOT TRENDING KEYWORDS TO CATEGORIES ---
STOP_KEYWORDS = {'ofertas', 'oferta', 'mercado libre', 'mercadolibre', 'cyber', 'cyberday', 'cyber day',
                 'black friday', 'regalo', 'regalos', 'liquidacion', 'outlet', 'envio gratis'}

def map_keyword(kw):
    code_, res = ml.get(f'/sites/{SITE}/domain_discovery/search', params={'q': kw, 'limit': 1}, auth=False)
    return kw, (res[0]['category_id'] if code_ == 200 and res else None)

def path_ids(cid):
    code_, d = ml.get(f'/categories/{cid}', auth=False)
    return cid, ([p['id'] for p in d.get('path_from_root', [])] if d else None)

root_ids = tree[tree.depth == 1].category_id.tolist()
kw_today = pd.read_sql(f"SELECT DISTINCT keyword FROM trends WHERE snapshot_date=? AND scope IN "
                       f"({','.join('?'*(len(root_ids)+1))})", conn, params=[TODAY, 'SITE', *root_ids]).keyword
cached = {r[0] for r in conn.execute("SELECT keyword FROM keyword_map")}
todo = [k for k in kw_today if k not in cached and k not in STOP_KEYWORDS]
mapped = ml.pmap(map_keyword, todo, label='keyword→category')
with DB_LOCK:
    conn.executemany("INSERT OR REPLACE INTO keyword_map VALUES (?,?,?)", [(k, c, TODAY) for k, c in mapped])
    conn.commit()

known = {r[0] for r in conn.execute("SELECT category_id FROM category_paths")}
need = [c for (c,) in conn.execute("SELECT DISTINCT category_id FROM keyword_map WHERE category_id IS NOT NULL")
        if c not in known]
paths = ml.pmap(path_ids, need, label='category paths')
with DB_LOCK:
    conn.executemany("INSERT OR REPLACE INTO category_paths VALUES (?,?)", [(c, json.dumps(p)) for c, p in paths if p])
    conn.commit()
print(f"🗺️  {len(kw_today)} distinct site/root keywords today | {len(todo)} newly mapped | {ml.calls:,} API calls so far")

## 3 · Feature engineering

In [ ]:
# --- CELL 6: FEATURES PER CATEGORY ---
def snapshot_days():
    return [r[0] for r in conn.execute("SELECT DISTINCT snapshot_date FROM trend_status ORDER BY 1")]

def ancestor_at(path, depth):
    """Ancestor id at `depth` (1-based); if the path is shallower, the deepest node."""
    return path[min(depth, len(path)) - 1] if path else None

def build_features(day, depth=ANALYSIS_DEPTH, site_weight=2.0):
    cats = pd.read_sql("SELECT * FROM categories WHERE snapshot_date=?", conn, params=(day,))
    status = pd.read_sql("SELECT scope, n_keywords FROM trend_status WHERE snapshot_date=?", conn, params=(day,))
    tr = pd.read_sql("SELECT scope, keyword, position FROM trends WHERE snapshot_date=?", conn, params=(day,))
    kmap = pd.read_sql("SELECT k.keyword, p.path_ids FROM keyword_map k JOIN category_paths p USING (category_id)", conn)
    kmap['target'] = kmap.path_ids.map(lambda s: ancestor_at(json.loads(s), depth))

    f = cats[cats.depth == depth].copy()
    names = cats.set_index('category_id')['name']
    f['root_name'], f['l2_name'] = f.root_id.map(names), f.l2_id.map(names)

    # 1) trend depth (0–50)
    f['trend_depth'] = f.category_id.map(status.set_index('scope')['n_keywords']).fillna(0)

    # 2) site + root keyword scores, credited to the category buyers land in
    root_ids = set(cats[cats.depth == 1].category_id)
    t = tr[tr.scope.isin(root_ids | {'SITE'})].merge(kmap[['keyword', 'target']], on='keyword')
    t = t[~t.keyword.isin(STOP_KEYWORDS)]
    t['w'] = 1 / np.sqrt(t.position)
    f['site_trend_score'] = f.category_id.map(t[t.scope == 'SITE'].groupby('target')['w'].sum()).fillna(0)
    f['root_trend_score'] = f.category_id.map(t[t.scope != 'SITE'].groupby('target')['w'].sum()).fillna(0)
    own = tr[tr.position <= 4].sort_values('position').groupby('scope')['keyword'].apply(', '.join)
    f['top_keywords'] = f.category_id.map(own).fillna('')                     # the category's own top searches
    f['won_keywords'] = f.category_id.map(t.sort_values('position').groupby('target')['keyword']
                                          .apply(lambda s: ', '.join(list(dict.fromkeys(s))[:3]))).fillna('')  # site/root hits

    # 3) supply
    f['log_listings'] = np.log1p(f.total_items.fillna(0))
    # demand vs supply: residual of trend_depth after a linear fit on log_listings
    mask = f.log_listings > 0
    if mask.sum() > 10:
        b, a = np.polyfit(f.loc[mask, 'log_listings'], f.loc[mask, 'trend_depth'], 1)
        f['demand_vs_supply'] = f.trend_depth - (a + b * f.log_listings)
    else:
        f['demand_vs_supply'] = 0.0

    # 4) dynamics vs previous snapshot (needs ≥ 2 days)
    prev = [d for d in snapshot_days() if d < day]
    f['listing_growth'], f['keyword_turnover'] = 0.0, np.nan
    if prev:
        p = prev[-1]
        base = pd.read_sql("SELECT category_id, total_items FROM categories WHERE snapshot_date=?", conn,
                           params=(p,)).set_index('category_id')['total_items']
        gap = max((pd.Timestamp(day) - pd.Timestamp(p)).days, 1)
        f['listing_growth'] = ((f.total_items - f.category_id.map(base)) / f.category_id.map(base).replace(0, np.nan)
                               / gap).fillna(0)
        prev_kw = pd.read_sql("SELECT scope, keyword FROM trends WHERE snapshot_date=?", conn, params=(p,))
        pk = prev_kw.groupby('scope')['keyword'].apply(set)
        ck = tr.groupby('scope')['keyword'].apply(set)
        f['keyword_turnover'] = f.category_id.map(
            {c: len(ck[c] - pk.get(c, set())) / len(ck[c]) for c in ck.index if c in pk.index and len(ck[c])})
    f['keyword_turnover'] = f['keyword_turnover'].fillna(f['keyword_turnover'].median() if f['keyword_turnover'].notna().any() else 0)
    return f, dict(n_days=len(prev) + 1, mapped_keywords=len(t))

feat, META = build_features(TODAY)
print(f"Features for {len(feat):,} level-{ANALYSIS_DEPTH} categories | snapshots: {META['n_days']} | "
      f"mapped site/root keyword hits: {META['mapped_keywords']}")

In [ ]:
# --- CELL 7: DEMAND SCORE & RANKING ---
WEIGHTS = {                     # percentile-ranked, so units don't matter; tweak freely
    'trend_depth':      0.35,   # breadth of buyer searches inside the category
    'root_trend_score': 0.25,   # wins in its vertical's top-50
    'site_trend_score': 0.20,   # wins in the whole site's top-50
    'log_listings':     0.10,   # market size (supply)
    'listing_growth':   0.10,   # sellers entering (day 2+)
}
w = dict(WEIGHTS)
if META['n_days'] < 2:
    w['trend_depth'] += w.pop('listing_growth')

def pct(s):                       # zeros stay at the bottom instead of sharing a mid rank
    return s.rank(pct=True, method='max').where(s > 0, 0) if (s == 0).mean() > 0.3 else s.rank(pct=True)

feat['demand_score'] = sum(pct(feat[k]) * v for k, v in w.items()) / sum(w.values()) * 100
top = feat.sort_values('demand_score', ascending=False)

show = ['root_name', 'l2_name', 'name', 'demand_score', 'trend_depth', 'site_trend_score', 'root_trend_score',
        'total_items', 'demand_vs_supply', 'top_keywords', 'won_keywords']
print("🔥 Top 30 categories by buyer demand")
display(top[show].head(30).style.format({'demand_score': '{:.1f}', 'trend_depth': '{:.0f}', 'site_trend_score': '{:.2f}',
                                         'root_trend_score': '{:.2f}', 'total_items': '{:,.0f}', 'demand_vs_supply': '{:+.1f}'}))

HOT = feat.demand_score.quantile(0.90)            # 'hot' = top 10% of categories
print(f"\n📊 Roll-up by level-2 category (top 15) | hot = demand score ≥ {HOT:.0f} (top 10%)")
l2 = (feat.groupby(['root_name', 'l2_name'])
          .agg(subcats=('name', 'size'), mean_score=('demand_score', 'mean'), max_score=('demand_score', 'max'),
               hot_subcats=('demand_score', lambda s: (s >= HOT).sum()), site_hits=('site_trend_score', 'sum'),
               listings=('total_items', 'sum'))
          .sort_values(['hot_subcats', 'mean_score'], ascending=False))
display(l2.head(15).style.format({'mean_score': '{:.1f}', 'max_score': '{:.1f}', 'site_hits': '{:.2f}', 'listings': '{:,.0f}'}))

print("\n💡 Most under-supplied demand (trend depth far above what listing count predicts)")
display(feat[feat.trend_depth >= 20].nlargest(15, 'demand_vs_supply')[['root_name', 'name', 'trend_depth', 'total_items',
        'demand_vs_supply', 'top_keywords']].style.format({'total_items': '{:,.0f}', 'demand_vs_supply': '{:+.1f}'}))

## 4 · Unsupervised profiling: PCA + k-means
Group categories into market profiles and check what the main axes of variation are.

In [ ]:
# --- CELL 8: PCA + K-MEANS ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

FEATURES = ['trend_depth', 'site_trend_score', 'root_trend_score', 'log_listings', 'demand_vs_supply',
            'listing_growth', 'keyword_turnover']
sparse = [c for c in FEATURES if (feat[c] != 0).mean() < 0.05]      # e.g. site_trend_score: <5% non-zero
FEATURES = [c for c in FEATURES if feat[c].std() > 0 and c not in sparse]
if sparse:
    print(f"Left out of clustering (non-zero in <5% of categories): {sparse}")
X = feat[FEATURES].copy()
for c in ['site_trend_score', 'root_trend_score']:
    if c in X:
        X[c] = np.log1p(X[c])                              # heavy-tailed
X = X.clip(X.quantile(0.01), X.quantile(0.99), axis=1)   # cap outliers so one giant doesn't get its own cluster
Xs = StandardScaler().fit_transform(X)

MIN_K, MAX_K = 4, 8          # k=3 tends to win silhouette but hides useful profiles
sample = np.random.RandomState(0).choice(len(Xs), min(len(Xs), 2000), replace=False)
scores = {k: silhouette_score(Xs[sample], KMeans(k, n_init=10, random_state=42).fit_predict(Xs)[sample])
          for k in range(MIN_K, MAX_K + 1)}
K = max(scores, key=scores.get)
km = KMeans(K, n_init=10, random_state=42).fit(Xs)
feat['cluster'] = km.labels_
print("Silhouette by k:", {k: round(v, 3) for k, v in scores.items()}, "→ k =", K)

prof = feat.groupby('cluster').agg(n=('name', 'size'), demand_score=('demand_score', 'mean'),
                                   trend_depth=('trend_depth', 'mean'), site=('site_trend_score', 'mean'),
                                   root=('root_trend_score', 'mean'), listings=('total_items', 'median'),
                                   dvs=('demand_vs_supply', 'mean'))
order = prof.demand_score.sort_values(ascending=False).index
med_listings, mean_depth = feat.total_items.median(), feat.trend_depth.mean()
names, used = {}, {}
for i, c in enumerate(order):
    p = prof.loc[c]
    if i == 0:
        n = 'High demand (site/vertical winners)'
    elif i == len(order) - 1:
        n = 'Low demand / long tail'
    elif p.listings < med_listings * 3 and p.dvs > 10:
        n = 'Under-supplied demand'
    elif p.listings > med_listings * 10 and p.trend_depth < mean_depth:
        n = 'Saturated supply'
    elif p.trend_depth >= mean_depth:
        n = 'Active demand, wins vertical top searches' if p.root > feat.root_trend_score.mean() * 2 else 'Active demand, broad searches'
    elif p.root > feat.root_trend_score.mean() * 2:
        n = 'Targeted demand (few searches, but top ones)'
    else:
        n = 'Moderate demand'
    used[n] = used.get(n, 0) + 1
    names[c] = n if used[n] == 1 else f'{n} #{used[n]}'
feat['profile'] = feat.cluster.map(names)
prof.index = prof.index.map(names)
display(prof.sort_values('demand_score', ascending=False).style.format(
    {'demand_score': '{:.1f}', 'trend_depth': '{:.1f}', 'site': '{:.2f}', 'root': '{:.2f}', 'listings': '{:,.0f}', 'dvs': '{:+.1f}'}))

pca = PCA(2).fit(Xs); P = pca.transform(Xs)
fig, ax = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2, 1]})
for name, g in feat.assign(pc1=P[:, 0], pc2=P[:, 1]).groupby('profile'):
    ax[0].scatter(g.pc1, g.pc2, s=10, alpha=.55, label=f"{name} ({len(g)})")
for _, r in feat.assign(pc1=P[:, 0], pc2=P[:, 1]).nlargest(12, 'demand_score').iterrows():
    ax[0].annotate(r['name'][:26], (r.pc1, r.pc2), fontsize=7)
ax[0].set(title=f'Category profiles (PCA, {pca.explained_variance_ratio_.sum():.0%} of variance)', xlabel='PC1', ylabel='PC2')
ax[0].legend(fontsize=8, markerscale=2)
pd.DataFrame(pca.components_.T, index=FEATURES, columns=['PC1', 'PC2']).plot.barh(ax=ax[1], title='PCA loadings')
plt.tight_layout(); plt.show()

pc1 = pd.Series(P[:, 0], index=feat.index)
print(f"Data-driven check: corr(PC1, demand_score) = {abs(np.corrcoef(pc1, feat.demand_score)[0, 1]):.2f}")

## 5 · Drill down to leaf niches
Fetch trends one level deeper for the top categories to find the exact niches buyers search for.

In [ ]:
# --- CELL 9: DRILL-DOWN ---
DRILL_TOP_N = 10

def children(cid):
    code_, d = ml.get(f'/categories/{cid}', auth=False)
    return [(cid, k['id'], k['name'].strip(), k.get('total_items_in_this_category')) for k in (d or {}).get('children_categories', [])]

kids = [k for ks in ml.pmap(children, top.head(DRILL_TOP_N).category_id.tolist()) for k in ks]
if kids:
    depth_by_kid = dict((s, len(k) if k is not None else 0) for s, k in ml.pmap(fetch_trends, [k[1] for k in kids]))
    niches = pd.DataFrame(kids, columns=['parent_id', 'niche_id', 'niche', 'listings'])
    niches['parent'] = niches.parent_id.map(top.set_index('category_id')['name'])
    niches['trend_depth'] = niches.niche_id.map(depth_by_kid)
    print("🎯 Niches inside the top categories")
    display(niches.sort_values(['trend_depth', 'listings'], ascending=False)
                  [['parent', 'niche', 'trend_depth', 'listings']].head(25).style.format({'listings': '{:,.0f}'}))
else:
    print("Top categories are already leaves.")

## 6 · Momentum (needs ≥ 2 daily runs)

In [ ]:
# --- CELL 10: MOMENTUM ACROSS SNAPSHOTS ---
days = snapshot_days()
if len(days) < 2:
    print(f"Only {len(days)} snapshot so far. Run again tomorrow to unlock growth, keyword turnover and momentum.")
else:
    hist = pd.concat([build_features(d)[0][['category_id', 'root_name', 'name', 'trend_depth', 'site_trend_score',
                                            'root_trend_score', 'total_items']].assign(day=d) for d in days[-14:]])
    first, last = days[-14:][0], days[-1]
    a = hist[hist.day == first].set_index('category_id'); b = hist[hist.day == last].set_index('category_id')
    mom = b[['root_name', 'name']].copy()
    mom['trend_depth_change'] = b.trend_depth - a.trend_depth.reindex(b.index)
    mom['trend_score_change'] = (b.site_trend_score + b.root_trend_score) - (a.site_trend_score + a.root_trend_score).reindex(b.index)
    mom['listing_growth'] = (b.total_items - a.total_items.reindex(b.index)) / a.total_items.reindex(b.index)
    if mom[['trend_depth_change', 'trend_score_change', 'listing_growth']].abs().sum().sum() == 0:
        print(f"No changes between {first} and {last} yet (snapshots too close together). Check again tomorrow.")
    print(f"📈 Rising demand ({first} → {last})")
    display(mom.sort_values(['trend_score_change', 'trend_depth_change'], ascending=False).head(15)
               .style.format({'trend_depth_change': '{:+.0f}', 'trend_score_change': '{:+.2f}', 'listing_growth': '{:+.1%}'}))
    print("\n🌱 Fastest supply growth (sellers entering)")
    display(mom[b.total_items > 500].nlargest(15, 'listing_growth').style.format(
        {'trend_depth_change': '{:+.0f}', 'trend_score_change': '{:+.2f}', 'listing_growth': '{:+.1%}'}))

In [ ]:
# --- CELL 11: EXPORT ---
out = f'{BASE_DIR}/mlc_category_demand_{TODAY}.csv'
feat.sort_values('demand_score', ascending=False).to_csv(out, index=False)
print(f"💾 Saved {out} | API calls this run: {ml.calls:,}")

### Caveats
* **Demand ≠ sales.** Trending searches show buyer interest. MercadoLibre's best-seller and sales data stays blocked for this app (403). If `/highlights` ever opens up (the probe in cell 2 checks every run), it could be added as a sales-side feature.
* `domain_discovery` sends each keyword to its most likely category. Ambiguous words ("gato", "bar") can land in the wrong one; `STOP_KEYWORDS` filters the most generic ones.
* `trend_depth` is capped at 50, so the biggest categories tie. `site_trend_score` and `root_trend_score` separate them.